# VidyaAI: Fine-tuning Gemma 4 with Unsloth

Fine-tune Gemma 4 E4B for multilingual Indian education using Unsloth + QLoRA.

**Requirements:** Google Colab with T4 GPU (free tier works)

## Setup

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 1. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
MODEL_NAME = "google/gemma-4-e4b"
MAX_SEQ_LENGTH = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True  # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

## 2. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimization
    random_state=42,
)

# Print trainable parameters
model.print_trainable_parameters()

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import json

# Gemma 4 chat template
def format_example(example):
    """Format example into Gemma 4 chat format."""
    text = (
        f"<start_of_turn>system\n{example['instruction']}<end_of_turn>\n"
        f"<start_of_turn>user\n{example['input']}<end_of_turn>\n"
        f"<start_of_turn>model\n{example['output']}<end_of_turn>"
    )
    return {"text": text}


# Load dataset - either from local file or HuggingFace
# Option 1: From local JSONL (upload train.jsonl to Colab)
# dataset = Dataset.from_json("train.jsonl")

# Option 2: Generate seed dataset inline
SEED_DATA = [
    {
        "instruction": "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।",
        "input": "प्रकाश संश्लेषण क्या है?",
        "output": "प्रकाश संश्लेषण वह प्रक्रिया है जिसमें हरे पौधे सूर्य के प्रकाश की ऊर्जा का उपयोग करके कार्बन डाइऑक्साइड और पानी से ग्लूकोज बनाते हैं। इस प्रक्रिया में ऑक्सीजन भी निकलती है।\n\nसमीकरण: 6CO₂ + 6H₂O + सूर्य का प्रकाश → C₆H₁₂O₆ + 6O₂\n\nयह प्रक्रिया पत्तियों में मौजूद क्लोरोफिल की मदद से होती है।"
    },
    {
        "instruction": "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।",
        "input": "पाइथागोरस प्रमेय क्या है? उदाहरण के साथ समझाइए।",
        "output": "पाइथागोरस प्रमेय कहता है कि एक समकोण त्रिभुज में, कर्ण का वर्ग अन्य दो भुजाओं के वर्गों के योग के बराबर होता है।\n\nसूत्र: c² = a² + b²\n\nउदाहरण: अगर दो भुजाएं 3 cm और 4 cm हैं:\nc² = 3² + 4² = 9 + 16 = 25\nc = √25 = 5 cm"
    },
    {
        "instruction": "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।",
        "input": "न्यूटन के गति के नियम बताइए।",
        "output": "न्यूटन के तीन नियम:\n\n1. जड़त्व का नियम: वस्तु अपनी अवस्था में रहती है जब तक बाहरी बल न लगे। उदाहरण: बस रुकने पर आगे गिरना।\n\n2. F = ma: बल = द्रव्यमान × त्वरण। उदाहरण: भारी गेंद को फेंकने में ज़्यादा बल लगता है।\n\n3. क्रिया-प्रतिक्रिया: हर क्रिया की बराबर और विपरीत प्रतिक्रिया होती है। उदाहरण: रॉकेट का उड़ना।"
    },
    {
        "instruction": "You are VidyaAI, an intelligent education assistant. Answer according to NCERT curriculum in clear English.",
        "input": "What is the water cycle?",
        "output": "The water cycle (hydrological cycle) is the continuous movement of water on, above, and below Earth's surface.\n\nStages:\n1. **Evaporation**: Sun heats water in oceans/rivers, turning it into water vapor.\n2. **Condensation**: Water vapor rises, cools, and forms clouds.\n3. **Precipitation**: Water falls as rain, snow, or hail.\n4. **Collection**: Water flows into rivers, lakes, and oceans.\n\nThis cycle is essential for life and has been happening for billions of years!"
    },
    {
        "instruction": "You are VidyaAI, an intelligent education assistant. Answer according to NCERT curriculum in clear English.",
        "input": "Explain the concept of democracy.",
        "output": "Democracy is a form of government where the people have the power to choose their leaders through free and fair elections.\n\nKey features:\n1. **Universal adult suffrage**: Every adult citizen has the right to vote.\n2. **Rule of law**: Everyone is equal before the law.\n3. **Fundamental rights**: Citizens have basic rights like freedom of speech.\n4. **Independent judiciary**: Courts work independently of the government.\n\nIndia is the world's largest democracy. Our Constitution, adopted on January 26, 1950, guarantees democratic governance."
    }
]

# For demo, use seed data. For full training, upload train.jsonl
dataset = Dataset.from_list(SEED_DATA)
dataset = dataset.map(format_example)

print(f"Dataset size: {len(dataset)}")
print(f"\nSample:\n{dataset[0]['text'][:500]}")

## 4. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memory reserved: {start_gpu_memory} GB / {max_memory} GB")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"Training completed in {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Test the Fine-tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_questions = [
    "प्रकाश संश्लेषण क्या है?",
    "भारत के पहले प्रधानमंत्री कौन थे?",
    "What is the formula for area of a circle?",
    "गुरुत्वाकर्षण बल क्या है?",
]

for q in test_questions:
    prompt = (
        "<start_of_turn>system\n"
        "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।"
        "<end_of_turn>\n"
        f"<start_of_turn>user\n{q}<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response}")

## 6. Save & Upload to HuggingFace

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("vidyaai-gemma4-lora")
tokenizer.save_pretrained("vidyaai-gemma4-lora")
print("Saved LoRA adapter to vidyaai-gemma4-lora/")

In [ ]:
# Upload to HuggingFace Hub
# First: huggingface-cli login
HF_REPO = "yashkuceriya/vidyaai-gemma4-lora"  # Change to your HF username

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Uploaded to https://huggingface.co/{HF_REPO}")

In [ ]:
# Optional: Save as GGUF for Ollama deployment
model.save_pretrained_gguf(
    "vidyaai-gemma4-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved GGUF model for Ollama deployment")